# Day 1-01｜球場 Keypoint 配對與 Homography 標定
> Python 籃球運動資料分析課程  
> 本單元建立相機畫面與 BEV 球場之間的對應點，並計算 Homography 投影矩陣。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 在 BEV 球場圖選擇具名 keypoint。
- 在相機畫面點選同一個實體位置，建立 keypoint pair。
- 使用至少四組 pair 計算 Homography。
- 檢查 footpoint 投影到 BEV 後的位置是否合理。

## 完成產出
- `assets/results/d1_01_bev_keypoint_reference.png`
- `assets/results/d1_01_homography_pairs.png`
- `assets/results/d1_01_projected_footpoint.png`


## 課程流程
1. 初始化 Colab 或本機課程環境。
2. 產生 BEV keypoint 參考圖。
3. 使用 pair 標定工具建立 BEV/camera 對應點。
4. 計算 Homography 並檢查投影結果。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


In [ ]:
from src.cv_utils import save_image_rgb
from src.geometry_utils import render_bev_court
from src.yolo_utils import (
    COURT_KEYPOINT_DISPLAY_NAMES,
    bev_court_bounds,
    court_vertices_bev_in_bounds,
)

CAMERA_IMAGE_PATH = COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png"
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"
BEV_IMAGE_PATH = COURSE_ROOT / "assets" / "results" / "d1_01_bev_keypoint_reference.png"

bev = render_bev_court(BEV_SPEC_PATH)
save_image_rgb(BEV_IMAGE_PATH, bev)

bev_points_all = court_vertices_bev_in_bounds(bev_court_bounds(BEV_SPEC_PATH))
bev_keypoints = [
    {"name": name, "xy": [float(x), float(y)]}
    for name, (x, y) in zip(COURT_KEYPOINT_DISPLAY_NAMES, bev_points_all)
]

print("camera image:", CAMERA_IMAGE_PATH)
print("BEV keypoint reference:", BEV_IMAGE_PATH)
print("keypoints:", len(bev_keypoints))


In [ ]:
from src.notebook_widgets import show_court_keypoint_pair_tool

show_court_keypoint_pair_tool(
    bev_image_path=BEV_IMAGE_PATH,
    camera_image_path=CAMERA_IMAGE_PATH,
    keypoints=bev_keypoints,
    canvas_width=1000,
    title="BEV / Camera Keypoint Pairing",
)


將工具輸出的 JSON 貼到下一格的 `pair_data`。本機或自動驗證時，下一格會先使用內建範例 pair。

In [ ]:
from src.geometry_utils import default_homography_pairs

# 使用互動工具時，請用工具輸出的完整 JSON 取代這個預設值。
pair_data = {"pairs": default_homography_pairs()}

pairs = pair_data["pairs"]
if len(pairs) < 4:
    raise ValueError("Homography 至少需要四組 BEV/camera pair。")

camera_points = [pair["camera_xy"] for pair in pairs]
bev_points = [pair["bev_xy"] for pair in pairs]
point_names = [pair["keypoint_name"] for pair in pairs]

for pair in pairs:
    print(pair["keypoint_name"], "camera=", pair["camera_xy"], "bev=", pair["bev_xy"])


In [ ]:
import numpy as np

from src.cv_utils import draw_points, read_image_rgb, show_image, side_by_side
from src.geometry_utils import compute_homography

H = compute_homography(camera_points, bev_points)
print("Homography matrix H =")
print(np.round(H, 3))

frame = read_image_rgb(CAMERA_IMAGE_PATH)
bev = render_bev_court(BEV_SPEC_PATH)
frame_vis = draw_points(frame, camera_points, labels=point_names)
bev_vis = draw_points(bev, bev_points, labels=point_names)
combo = side_by_side(frame_vis, bev_vis)
show_image(combo, "Camera-BEV calibration pairs")

out_path = COURSE_ROOT / "assets" / "results" / "d1_01_homography_pairs.png"
save_image_rgb(out_path, combo)
print("saved:", out_path)


In [ ]:
from src.cv_utils import bottom_center, draw_boxes
from src.geometry_utils import DEFAULT_SAMPLE_PLAYER_BBOX_XYXY, project_points

bbox = DEFAULT_SAMPLE_PLAYER_BBOX_XYXY
footpoint = bottom_center(bbox)
bev_footpoint = project_points([footpoint], H)[0]

frame_vis = draw_boxes(frame, [bbox], ["sample_player"])
frame_vis = draw_points(frame_vis, [footpoint], ["footpoint"])
bev_vis = draw_points(bev, [bev_footpoint], ["projected_footpoint"])
combo = side_by_side(frame_vis, bev_vis)
show_image(combo, "Projected footpoint")

out_path = COURSE_ROOT / "assets" / "results" / "d1_01_projected_footpoint.png"
save_image_rgb(out_path, combo)
print("footpoint:", [round(v, 2) for v in footpoint])
print("BEV footpoint:", np.round(bev_footpoint, 2).tolist())
print("saved:", out_path)


Homography 只描述同一平面上的投影關係。籃球員 bbox 的底邊中心點是地面接觸位置的近似值；若框住身體的方式不一致，投影點也會跟著偏移。